In [45]:
import torch

# Convert feature arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train_np, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val_np, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_np, dtype=torch.float32)

# Convert labels to PyTorch tensors
y_train_tensor = torch.tensor(y_train_processed, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_processed, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_processed, dtype=torch.float32)

print("PyTorch tensors created successfully.")

print("\nTraining:")
print("X:", X_train_tensor.shape, X_train_tensor.dtype)
print("y:", y_train_tensor.shape, y_train_tensor.dtype)

print("\nValidation:")
print("X:", X_val_tensor.shape, X_val_tensor.dtype)
print("y:", y_val_tensor.shape, y_val_tensor.dtype)

print("\nTest:")
print("X:", X_test_tensor.shape, X_test_tensor.dtype)
print("y:", y_test_tensor.shape, y_test_tensor.dtype)

PyTorch tensors created successfully.

Training:
X: torch.Size([177576, 21]) torch.float32
y: torch.Size([177576]) torch.float32

Validation:
X: torch.Size([38052, 21]) torch.float32
y: torch.Size([38052]) torch.float32

Test:
X: torch.Size([38052, 21]) torch.float32
y: torch.Size([38052]) torch.float32


In [46]:
import torch.nn as nn

class DiabetesDNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            # First hidden layer
            nn.Linear(21, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Second hidden layer
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Third hidden layer
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),

            # Output layer
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

In [47]:
# Create the DNN model
model = DiabetesDNN()

print(model)

DiabetesDNN(
  (network): Sequential(
    (0): Linear(in_features=21, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
  )
)


In [48]:
# Count trainable parameters

total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", total_params)

print("\nParameters by layer:")
for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(f"{name:30s} {parameter.numel():,}")

Trainable parameters: 13185

Parameters by layer:
network.0.weight               2,688
network.0.bias                 128
network.3.weight               8,192
network.3.bias                 64
network.6.weight               2,048
network.6.bias                 32
network.9.weight               32
network.9.bias                 1


In [49]:
import numpy as np

# Count classes in the training set
class_counts = np.bincount(y_train_processed.astype(int))

# Calculate balanced class weights
num_samples = len(y_train_processed)
num_classes = len(class_counts)

class_weights = num_samples / (num_classes * class_counts)

print("Training class counts:")
print("Class 0:", class_counts[0])
print("Class 1:", class_counts[1])

print("\nCalculated class weights:")
print("Class 0:", class_weights[0])
print("Class 1:", class_weights[1])

Training class counts:
Class 0: 152834
Class 1: 24742

Calculated class weights:
Class 0: 0.5809440307784917
Class 1: 3.5885538760003235


In [50]:
import torch
import torch.nn as nn

# Convert the positive-class weight to a PyTorch tensor
pos_weight = torch.tensor(
    class_weights[1],
    dtype=torch.float32
)

# Weighted binary cross-entropy loss
criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

print("Weighted loss function created successfully.")
print("Positive-class weight:", pos_weight.item())
print("Loss function:", criterion)

Weighted loss function created successfully.
Positive-class weight: 3.5885539054870605
Loss function: BCEWithLogitsLoss()


In [51]:
import torch.optim as optim

# Adam optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Optimizer created successfully.")
print("Optimizer:", optimizer)
print("Learning rate:", 1e-3)

Optimizer created successfully.
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.001


In [52]:

from torch.utils.data import TensorDataset, DataLoader

# Create TensorDatasets
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

# Create DataLoaders
batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("DataLoaders created successfully.")

print("\nBatch size:", batch_size)
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders created successfully.

Batch size: 256
Training batches: 694
Validation batches: 149
Test batches: 149


In [53]:
# Inspect one batch from the training DataLoader

X_batch, y_batch = next(iter(train_loader))

print("ONE TRAINING BATCH")
print("=" * 60)

print("Feature batch shape:", X_batch.shape)
print("Feature data type:", X_batch.dtype)

print("\nLabel batch shape:", y_batch.shape)
print("Label data type:", y_batch.dtype)

print("\nFirst patient's features:")
print(X_batch[0])

print("\nFirst 10 labels:")
print(y_batch[:10])

ONE TRAINING BATCH
Feature batch shape: torch.Size([256, 21])
Feature data type: torch.float32

Label batch shape: torch.Size([256])
Label data type: torch.float32

First patient's features:
tensor([ 1.0000,  1.0000,  1.0000, -0.5110,  1.0000,  0.0000,  0.0000,  1.0000,
         1.0000,  1.0000,  0.0000,  1.0000,  0.0000,  1.0000, -0.4299, -0.4867,
         0.0000,  0.0000, -0.6660,  6.0000,  8.0000])

First 10 labels:
tensor([0., 0., 0., 0., 1., 0., 0., 0., 0., 0.])


In [54]:
# Put the model in training mode
model.train()

# Perform one forward pass
logits = model(X_batch)

print("FORWARD PASS")
print("=" * 60)

print("Input shape:", X_batch.shape)
print("Output shape:", logits.shape)

print("\nFirst 10 raw logits:")
print(logits[:10].squeeze())

# Convert logits to probabilities for inspection
probabilities = torch.sigmoid(logits)

print("\nFirst 10 probabilities:")
print(probabilities[:10].squeeze())

FORWARD PASS
Input shape: torch.Size([256, 21])
Output shape: torch.Size([256, 1])

First 10 raw logits:
tensor([-0.3025, -0.2532, -0.1269, -0.3719, -0.4149, -0.1179, -0.3922, -0.0715,
        -0.2819, -0.0582], grad_fn=<SqueezeBackward0>)

First 10 probabilities:
tensor([0.4250, 0.4370, 0.4683, 0.4081, 0.3977, 0.4706, 0.4032, 0.4821, 0.4300,
        0.4855], grad_fn=<SqueezeBackward0>)


In [55]:
# Calculate the initial loss for the current batch

# Remove the final dimension from logits
# [256, 1] → [256]
batch_logits = logits.squeeze(1)

# Calculate weighted binary cross-entropy loss
initial_loss = criterion(batch_logits, y_batch)

print("INITIAL LOSS")
print("=" * 60)
print("Loss:", initial_loss.item())

INITIAL LOSS
Loss: 1.0358978509902954


In [56]:
# Perform one complete training step

# Make sure the model is in training mode
model.train()

# Clear gradients from any previous step
optimizer.zero_grad()

# Forward pass
logits = model(X_batch)

# Remove the final dimension: [256, 1] → [256]
batch_logits = logits.squeeze(1)

# Calculate loss
loss = criterion(batch_logits, y_batch)

# Backpropagation
loss.backward()

# Update model parameters
optimizer.step()

print("ONE TRAINING STEP COMPLETED")
print("=" * 60)
print("Loss before parameter update:", loss.item())

ONE TRAINING STEP COMPLETED
Loss before parameter update: 1.0345157384872437


In [57]:
# Check whether the model parameters were updated

print("PARAMETER UPDATE CHECK")
print("=" * 60)

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(
            f"{name:30s} | "
            f"mean = {parameter.data.mean().item():.6f} | "
            f"grad mean = {parameter.grad.mean().item():.6f}"
        )

PARAMETER UPDATE CHECK
network.0.weight               | mean = -0.002806 | grad mean = 0.000032
network.0.bias                 | mean = -0.004851 | grad mean = 0.000022
network.3.weight               | mean = 0.000527 | grad mean = 0.000081
network.3.bias                 | mean = 0.011600 | grad mean = 0.000256
network.6.weight               | mean = 0.003540 | grad mean = 0.000058
network.6.bias                 | mean = 0.010312 | grad mean = 0.000484
network.9.weight               | mean = -0.027019 | grad mean = 0.002310
network.9.bias                 | mean = -0.062738 | grad mean = -0.006263


In [58]:
# Reset the model before real training

model = DiabetesDNN()

# Recreate the weighted loss
pos_weight = torch.tensor(
    class_weights[1],
    dtype=torch.float32
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

# Recreate the optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=1e-3
)

print("Model reset successfully.")
print("\nArchitecture:")
print(model)

print("\nTrainable parameters:")
print(sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
))

print("\nOptimizer:")
print(optimizer.__class__.__name__)

print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Positive-class weight:", pos_weight.item())

Model reset successfully.

Architecture:
DiabetesDNN(
  (network): Sequential(
    (0): Linear(in_features=21, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.3, inplace=False)
    (9): Linear(in_features=32, out_features=1, bias=True)
  )
)

Trainable parameters:
13185

Optimizer:
Adam
Learning rate: 0.001
Positive-class weight: 3.5885539054870605


In [59]:
from sklearn.metrics import roc_auc_score
import copy
import numpy as np
import torch

# Training configuration
max_epochs = 100
patience = 10

# Store training history
history = {
    "train_loss": [],
    "val_loss": [],
    "val_auc": []
}

# Early stopping variables
best_val_auc = -np.inf
epochs_without_improvement = 0
best_model_state = None

print("Starting DNN training...")
print("=" * 70)

for epoch in range(max_epochs):

    # ============================================================
    # TRAINING
    # ============================================================

    model.train()

    running_train_loss = 0.0
    train_samples = 0

    for X_batch, y_batch in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        logits = model(X_batch).squeeze(1)

        # Calculate weighted loss
        loss = criterion(logits, y_batch)

        # Backpropagation
        loss.backward()

        # Update model parameters
        optimizer.step()

        # Accumulate loss
        batch_size_actual = X_batch.size(0)
        running_train_loss += loss.item() * batch_size_actual
        train_samples += batch_size_actual

    epoch_train_loss = running_train_loss / train_samples

    # ============================================================
    # VALIDATION
    # ============================================================

    model.eval()

    running_val_loss = 0.0
    val_samples = 0

    val_probabilities = []
    val_true_labels = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            # Forward pass
            logits = model(X_batch).squeeze(1)

            # Validation loss
            loss = criterion(logits, y_batch)

            # Convert logits to probabilities
            probabilities = torch.sigmoid(logits)

            # Accumulate validation loss
            batch_size_actual = X_batch.size(0)
            running_val_loss += loss.item() * batch_size_actual
            val_samples += batch_size_actual

            # Store predictions and true labels
            val_probabilities.extend(
                probabilities.cpu().numpy()
            )

            val_true_labels.extend(
                y_batch.cpu().numpy()
            )

    epoch_val_loss = running_val_loss / val_samples

    # Calculate validation ROC-AUC
    epoch_val_auc = roc_auc_score(
        val_true_labels,
        val_probabilities
    )

    # Store history
    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)
    history["val_auc"].append(epoch_val_auc)

    # ============================================================
    # EARLY STOPPING / CHECKPOINTING
    # ============================================================

    if epoch_val_auc > best_val_auc:

        # Validation AUC improved
        best_val_auc = epoch_val_auc
        epochs_without_improvement = 0

        # Save a copy of the best model parameters
        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        checkpoint_status = " ← BEST"

    else:

        # No improvement
        epochs_without_improvement += 1

        checkpoint_status = (
            f" | patience: "
            f"{epochs_without_improvement}/{patience}"
        )

    # ============================================================
    # PRINT EPOCH RESULTS
    # ============================================================

    print(
        f"Epoch {epoch + 1:03d}/{max_epochs} | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f} | "
        f"Val ROC-AUC: {epoch_val_auc:.4f}"
        f"{checkpoint_status}"
    )

    # ============================================================
    # EARLY STOPPING
    # ============================================================

    if epochs_without_improvement >= patience:

        print("\nEarly stopping triggered.")
        print(
            f"No validation ROC-AUC improvement for "
            f"{patience} consecutive epochs."
        )

        break


# ================================================================
# RESTORE BEST MODEL
# ================================================================

if best_model_state is not None:

    model.load_state_dict(best_model_state)

    print("\nBest model restored successfully.")
    print(f"Best validation ROC-AUC: {best_val_auc:.4f}")

print("\nTraining completed.")

Starting DNN training...
Epoch 001/100 | Train Loss: 0.7015 | Val Loss: 0.6749 | Val ROC-AUC: 0.8221 ← BEST
Epoch 002/100 | Train Loss: 0.6756 | Val Loss: 0.6676 | Val ROC-AUC: 0.8243 ← BEST
Epoch 003/100 | Train Loss: 0.6735 | Val Loss: 0.6653 | Val ROC-AUC: 0.8254 ← BEST
Epoch 004/100 | Train Loss: 0.6719 | Val Loss: 0.6654 | Val ROC-AUC: 0.8254 ← BEST
Epoch 005/100 | Train Loss: 0.6687 | Val Loss: 0.6650 | Val ROC-AUC: 0.8263 ← BEST
Epoch 006/100 | Train Loss: 0.6682 | Val Loss: 0.6638 | Val ROC-AUC: 0.8264 ← BEST
Epoch 007/100 | Train Loss: 0.6680 | Val Loss: 0.6639 | Val ROC-AUC: 0.8261 | patience: 1/10
Epoch 008/100 | Train Loss: 0.6660 | Val Loss: 0.6639 | Val ROC-AUC: 0.8267 ← BEST
Epoch 009/100 | Train Loss: 0.6666 | Val Loss: 0.6631 | Val ROC-AUC: 0.8272 ← BEST
Epoch 010/100 | Train Loss: 0.6660 | Val Loss: 0.6623 | Val ROC-AUC: 0.8273 ← BEST
Epoch 011/100 | Train Loss: 0.6655 | Val Loss: 0.6656 | Val ROC-AUC: 0.8270 | patience: 1/10
Epoch 012/100 | Train Loss: 0.6652 | Val L

In [60]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Put the restored best model into evaluation mode
model.eval()

test_probabilities = []
test_true_labels = []

# Generate predictions without calculating gradients
with torch.no_grad():

    for X_batch, y_batch in test_loader:

        # Forward pass
        logits = model(X_batch).squeeze(1)

        # Convert logits to probabilities
        probabilities = torch.sigmoid(logits)

        # Store predictions and true labels
        test_probabilities.extend(
            probabilities.cpu().numpy()
        )

        test_true_labels.extend(
            y_batch.cpu().numpy()
        )

# Convert to NumPy arrays
test_probabilities = np.array(test_probabilities)
test_true_labels = np.array(test_true_labels)

print("TEST PREDICTIONS GENERATED")
print("=" * 60)

print("Number of predictions:", len(test_probabilities))
print("Number of true labels:", len(test_true_labels))

print("\nProbability range:")
print("Minimum:", test_probabilities.min())
print("Maximum:", test_probabilities.max())

print("\nFirst 10 test probabilities:")
print(test_probabilities[:10])

print("\nFirst 10 true labels:")
print(test_true_labels[:10])

TEST PREDICTIONS GENERATED
Number of predictions: 38052
Number of true labels: 38052

Probability range:
Minimum: 1.315385e-05
Maximum: 0.95774096

First 10 test probabilities:
[0.50625366 0.12168752 0.5937284  0.45701844 0.24997047 0.1034712
 0.24494106 0.01843453 0.01760314 0.3879419 ]

First 10 true labels:
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]


In [61]:
# ============================================================
# FINAL TEST METRICS
# ============================================================

# Classification threshold
threshold = 0.5

# Convert probabilities into binary predictions
test_predictions = (
    test_probabilities >= threshold
).astype(int)

# ------------------------------------------------------------
# Threshold-independent metrics
# ------------------------------------------------------------

test_roc_auc = roc_auc_score(
    test_true_labels,
    test_probabilities
)

test_pr_auc = average_precision_score(
    test_true_labels,
    test_probabilities
)

# ------------------------------------------------------------
# Threshold-dependent metrics
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    test_true_labels,
    test_predictions
)

test_precision = precision_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

# ------------------------------------------------------------
# Confusion matrix
# ------------------------------------------------------------

test_cm = confusion_matrix(
    test_true_labels,
    test_predictions
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("FINAL DNN TEST RESULTS")
print("=" * 60)

print(f"ROC-AUC   : {test_roc_auc:.4f}")
print(f"PR-AUC    : {test_pr_auc:.4f}")
print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1-score  : {test_f1:.4f}")

print("\nConfusion Matrix:")
print(test_cm)

print("\nConfusion Matrix Interpretation:")
print(f"TN = {test_cm[0, 0]}")
print(f"FP = {test_cm[0, 1]}")
print(f"FN = {test_cm[1, 0]}")
print(f"TP = {test_cm[1, 1]}")

FINAL DNN TEST RESULTS
ROC-AUC   : 0.8295
PR-AUC    : 0.4232
Accuracy  : 0.7909
Precision : 0.3615
Recall    : 0.6539
F1-score  : 0.4656

Confusion Matrix:
[[26627  6123]
 [ 1835  3467]]

Confusion Matrix Interpretation:
TN = 26627
FP = 6123
FN = 1835
TP = 3467


In [62]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class UA_DNN(nn.Module):

    def __init__(self, input_dim):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.drop1 = nn.Dropout(0.3)

        self.fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.drop2 = nn.Dropout(0.3)

        self.fc3 = nn.Linear(64, 32)
        self.drop3 = nn.Dropout(0.2)

        self.fc4 = nn.Linear(32, 1)

    def forward(self, x):

        x = self.fc1(x)
        x = self.bn1(x)
        x = F.relu(x)
        x = self.drop1(x)

        x = self.fc2(x)
        x = self.bn2(x)
        x = F.relu(x)
        x = self.drop2(x)

        x = self.fc3(x)
        x = F.relu(x)
        x = self.drop3(x)

        x = self.fc4(x)

        return torch.sigmoid(x)


# Create the PRD-compliant model
model = UA_DNN(input_dim=21)

print("PRD-compliant DNN created successfully.")
print(model)

print("\nTrainable parameters:")
print(sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
))

PRD-compliant DNN created successfully.
UA_DNN(
  (fc1): Linear(in_features=21, out_features=128, bias=True)
  (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (drop1): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=64, bias=True)
  (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (drop2): Dropout(p=0.3, inplace=False)
  (fc3): Linear(in_features=64, out_features=32, bias=True)
  (drop3): Dropout(p=0.2, inplace=False)
  (fc4): Linear(in_features=32, out_features=1, bias=True)
)

Trainable parameters:
13569


In [63]:
# ============================================================
# PRD-COMPLIANT WEIGHTED BCELoss
# ============================================================

# Positive-class weight calculated from the training set
positive_weight = class_weights[1]

# Create per-sample weights
sample_weights = torch.where(
    y_train_tensor == 1.0,
    torch.tensor(positive_weight, dtype=torch.float32),
    torch.tensor(1.0, dtype=torch.float32)
)

# Weighted binary cross-entropy
criterion = nn.BCELoss(
    weight=sample_weights
)

print("PRD-compliant loss created successfully.")
print("Loss function:", criterion)
print("Positive-class weight:", positive_weight)

PRD-compliant loss created successfully.
Loss function: BCELoss()
Positive-class weight: 3.5885538760003235


In [64]:
# ============================================================
# PRD-COMPLIANT ADAM OPTIMIZER
# ============================================================

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("PRD-compliant optimizer created successfully.")
print("Optimizer:", optimizer)
print("Learning rate:", optimizer.param_groups[0]["lr"])

PRD-compliant optimizer created successfully.
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Learning rate: 0.001


In [65]:
# ============================================================
# FIXED PRD-COMPLIANT BCELoss
# ============================================================

criterion = nn.BCELoss(reduction='none')

positive_weight = float(class_weights[1])

print("Base BCELoss created successfully.")
print("Positive-class weight:", positive_weight)
print("Loss reduction: none")

Base BCELoss created successfully.
Positive-class weight: 3.5885538760003235
Loss reduction: none


In [66]:
# ============================================================
# PRD-COMPLIANT DNN TRAINING LOOP — FIXED WEIGHTING
# ============================================================

from sklearn.metrics import roc_auc_score
import copy
import numpy as np
import torch


MAX_EPOCHS = 100
PATIENCE = 10

best_val_auc = -np.inf
best_model_state = None
patience_counter = 0

train_losses = []
val_losses = []
val_auc_history = []

print("Starting PRD-compliant DNN training...")
print("=" * 70)


for epoch in range(1, MAX_EPOCHS + 1):

    # ========================================================
    # TRAINING
    # ========================================================

    model.train()

    running_train_loss = 0.0
    train_samples = 0

    for X_batch, y_batch in train_loader:

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        predictions = model(X_batch).squeeze(1)

        # Individual BCE losses
        individual_losses = criterion(
            predictions,
            y_batch
        )

        # Assign class weight to positive samples
        batch_weights = torch.where(
            y_batch == 1.0,
            torch.tensor(
                positive_weight,
                dtype=torch.float32
            ),
            torch.tensor(
                1.0,
                dtype=torch.float32
            )
        )

        # Apply class weights
        weighted_loss = (
            individual_losses * batch_weights
        ).mean()

        # Backpropagation
        weighted_loss.backward()

        # Update parameters
        optimizer.step()

        # Accumulate batch loss
        batch_size = X_batch.size(0)

        running_train_loss += (
            weighted_loss.item() * batch_size
        )

        train_samples += batch_size

    epoch_train_loss = (
        running_train_loss / train_samples
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model.eval()

    running_val_loss = 0.0
    val_samples = 0

    val_probabilities = []
    val_true_labels = []

    with torch.no_grad():

        for X_batch, y_batch in val_loader:

            # Forward pass
            predictions = model(
                X_batch
            ).squeeze(1)

            # Individual validation losses
            individual_losses = criterion(
                predictions,
                y_batch
            )

            # Same class weighting
            batch_weights = torch.where(
                y_batch == 1.0,
                torch.tensor(
                    positive_weight,
                    dtype=torch.float32
                ),
                torch.tensor(
                    1.0,
                    dtype=torch.float32
                )
            )

            weighted_val_loss = (
                individual_losses * batch_weights
            ).mean()

            # Accumulate validation loss
            batch_size = X_batch.size(0)

            running_val_loss += (
                weighted_val_loss.item() * batch_size
            )

            val_samples += batch_size

            # Store probabilities
            val_probabilities.extend(
                predictions.cpu().numpy()
            )

            val_true_labels.extend(
                y_batch.cpu().numpy()
            )


    epoch_val_loss = (
        running_val_loss / val_samples
    )


    # ========================================================
    # VALIDATION ROC-AUC
    # ========================================================

    epoch_val_auc = roc_auc_score(
        val_true_labels,
        val_probabilities
    )


    # ========================================================
    # STORE HISTORY
    # ========================================================

    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    val_auc_history.append(epoch_val_auc)


    # ========================================================
    # BEST CHECKPOINT
    # ========================================================

    if epoch_val_auc > best_val_auc:

        best_val_auc = epoch_val_auc

        best_model_state = copy.deepcopy(
            model.state_dict()
        )

        patience_counter = 0

        print(
            f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"Train Loss: {epoch_train_loss:.4f} | "
            f"Val Loss: {epoch_val_loss:.4f} | "
            f"Val ROC-AUC: {epoch_val_auc:.4f} "
            f"← BEST"
        )

    else:

        patience_counter += 1

        print(
            f"Epoch {epoch:03d}/{MAX_EPOCHS} | "
            f"Train Loss: {epoch_train_loss:.4f} | "
            f"Val Loss: {epoch_val_loss:.4f} | "
            f"Val ROC-AUC: {epoch_val_auc:.4f} "
            f"| patience: "
            f"{patience_counter}/{PATIENCE}"
        )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if patience_counter >= PATIENCE:

        print("\nEarly stopping triggered.")

        print(
            f"No validation ROC-AUC improvement "
            f"for {PATIENCE} consecutive epochs."
        )

        break


# ============================================================
# RESTORE BEST MODEL
# ============================================================

if best_model_state is not None:

    model.load_state_dict(
        best_model_state
    )

    print("\nBest model restored successfully.")

    print(
        f"Best validation ROC-AUC: "
        f"{best_val_auc:.4f}"
    )

print("\nTraining completed.")

Starting PRD-compliant DNN training...
Epoch 001/100 | Train Loss: 0.6863 | Val Loss: 0.6663 | Val ROC-AUC: 0.8253 ← BEST
Epoch 002/100 | Train Loss: 0.6708 | Val Loss: 0.6637 | Val ROC-AUC: 0.8274 ← BEST
Epoch 003/100 | Train Loss: 0.6689 | Val Loss: 0.6620 | Val ROC-AUC: 0.8274 ← BEST
Epoch 004/100 | Train Loss: 0.6655 | Val Loss: 0.6621 | Val ROC-AUC: 0.8276 ← BEST
Epoch 005/100 | Train Loss: 0.6654 | Val Loss: 0.6615 | Val ROC-AUC: 0.8279 ← BEST
Epoch 006/100 | Train Loss: 0.6643 | Val Loss: 0.6608 | Val ROC-AUC: 0.8283 ← BEST
Epoch 007/100 | Train Loss: 0.6634 | Val Loss: 0.6613 | Val ROC-AUC: 0.8281 | patience: 1/10
Epoch 008/100 | Train Loss: 0.6623 | Val Loss: 0.6616 | Val ROC-AUC: 0.8285 ← BEST
Epoch 009/100 | Train Loss: 0.6628 | Val Loss: 0.6623 | Val ROC-AUC: 0.8281 | patience: 1/10
Epoch 010/100 | Train Loss: 0.6623 | Val Loss: 0.6605 | Val ROC-AUC: 0.8282 | patience: 2/10
Epoch 011/100 | Train Loss: 0.6609 | Val Loss: 0.6610 | Val ROC-AUC: 0.8281 | patience: 3/10
Epoch 01

In [67]:
# ============================================================
# FINAL TEST EVALUATION — PRD-COMPLIANT DNN
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Make absolutely sure the best checkpoint is loaded
model.load_state_dict(best_model_state)

# Evaluation mode
model.eval()

test_probabilities = []
test_true_labels = []

# Generate final test probabilities
with torch.no_grad():

    for X_batch, y_batch in test_loader:

        predictions = model(
            X_batch
        ).squeeze(1)

        test_probabilities.extend(
            predictions.cpu().numpy()
        )

        test_true_labels.extend(
            y_batch.cpu().numpy()
        )

# Convert to NumPy
test_probabilities = np.array(
    test_probabilities
)

test_true_labels = np.array(
    test_true_labels
)

# ============================================================
# CLASSIFICATION THRESHOLD
# ============================================================

threshold = 0.5

test_predictions = (
    test_probabilities >= threshold
).astype(int)

# ============================================================
# METRICS
# ============================================================

test_roc_auc = roc_auc_score(
    test_true_labels,
    test_probabilities
)

test_pr_auc = average_precision_score(
    test_true_labels,
    test_probabilities
)

test_accuracy = accuracy_score(
    test_true_labels,
    test_predictions
)

test_precision = precision_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_recall = recall_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_f1 = f1_score(
    test_true_labels,
    test_predictions,
    zero_division=0
)

test_cm = confusion_matrix(
    test_true_labels,
    test_predictions
)

# ============================================================
# DISPLAY
# ============================================================

print("FINAL PRD-COMPLIANT DNN TEST RESULTS")
print("=" * 60)

print(f"ROC-AUC   : {test_roc_auc:.4f}")
print(f"PR-AUC    : {test_pr_auc:.4f}")
print(f"Accuracy  : {test_accuracy:.4f}")
print(f"Precision : {test_precision:.4f}")
print(f"Recall    : {test_recall:.4f}")
print(f"F1-score  : {test_f1:.4f}")

print("\nConfusion Matrix:")
print(test_cm)

print("\nConfusion Matrix Interpretation:")
print(f"TN = {test_cm[0, 0]}")
print(f"FP = {test_cm[0, 1]}")
print(f"FN = {test_cm[1, 0]}")
print(f"TP = {test_cm[1, 1]}")

print("\nProbability range:")
print(f"Minimum = {test_probabilities.min():.6f}")
print(f"Maximum = {test_probabilities.max():.6f}")

FINAL PRD-COMPLIANT DNN TEST RESULTS
ROC-AUC   : 0.8301
PR-AUC    : 0.4264
Accuracy  : 0.7872
Precision : 0.3578
Recall    : 0.6635
F1-score  : 0.4649

Confusion Matrix:
[[26437  6313]
 [ 1784  3518]]

Confusion Matrix Interpretation:
TN = 26437
FP = 6313
FN = 1784
TP = 3518

Probability range:
Minimum = 0.000108
Maximum = 0.936456


In [68]:
# ============================================================
# MC DROPOUT SETUP
# ============================================================

import torch.nn as nn

# Load the best DNN checkpoint again
model.load_state_dict(best_model_state)

# Start with the entire model in evaluation mode
model.eval()

# Activate ONLY dropout layers
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()

print("MC Dropout mode configured successfully.")
print("=" * 60)

# Verify the behavior of each layer
for name, module in model.named_modules():

    if isinstance(module, nn.Dropout):
        print(
            f"{name}: Dropout ACTIVE | "
            f"p = {module.p}"
        )

    elif isinstance(module, nn.BatchNorm1d):
        print(
            f"{name}: BatchNorm EVAL | "
            f"training = {module.training}"
        )

MC Dropout mode configured successfully.
bn1: BatchNorm EVAL | training = False
drop1: Dropout ACTIVE | p = 0.3
bn2: BatchNorm EVAL | training = False
drop2: Dropout ACTIVE | p = 0.3
drop3: Dropout ACTIVE | p = 0.2


In [69]:
# ============================================================
# MC DROPOUT INFERENCE — T = 50
# ============================================================

import numpy as np
import torch

# Number of stochastic forward passes
T = 50

# Make sure best checkpoint is loaded
model.load_state_dict(best_model_state)

# Evaluation mode first
model.eval()

# Activate ONLY Dropout layers
for module in model.modules():
    if isinstance(module, nn.Dropout):
        module.train()

# Store predictions from every stochastic pass
mc_predictions = []

print("Starting MC Dropout inference...")
print("=" * 60)
print(f"Number of stochastic passes (T): {T}")
print(f"Number of test samples: {len(test_true_labels)}")

# ------------------------------------------------------------
# Perform T stochastic forward passes
# ------------------------------------------------------------

with torch.no_grad():

    for t in range(T):

        pass_predictions = []

        for X_batch, _ in test_loader:

            predictions = model(
                X_batch
            ).squeeze(1)

            pass_predictions.extend(
                predictions.cpu().numpy()
            )

        pass_predictions = np.array(
            pass_predictions
        )

        mc_predictions.append(
            pass_predictions
        )

        print(
            f"Pass {t + 1:02d}/{T} completed | "
            f"Min: {pass_predictions.min():.6f} | "
            f"Max: {pass_predictions.max():.6f} | "
            f"Mean: {pass_predictions.mean():.6f}"
        )

# Convert to NumPy array
mc_predictions = np.array(
    mc_predictions
)

print("\nMC Dropout inference completed.")
print("=" * 60)

print(
    "MC prediction array shape:",
    mc_predictions.shape
)

print(
    "Expected shape:",
    (T, len(test_true_labels))
)

print(
    "Total stochastic predictions:",
    mc_predictions.size
)

Starting MC Dropout inference...
Number of stochastic passes (T): 50
Number of test samples: 38052
Pass 01/50 completed | Min: 0.000016 | Max: 0.972535 | Mean: 0.295636
Pass 02/50 completed | Min: 0.000008 | Max: 0.964224 | Mean: 0.295230
Pass 03/50 completed | Min: 0.000011 | Max: 0.969230 | Mean: 0.295780
Pass 04/50 completed | Min: 0.000002 | Max: 0.962963 | Mean: 0.295447
Pass 05/50 completed | Min: 0.000006 | Max: 0.969651 | Mean: 0.296115
Pass 06/50 completed | Min: 0.000006 | Max: 0.958834 | Mean: 0.295917
Pass 07/50 completed | Min: 0.000004 | Max: 0.977805 | Mean: 0.295860
Pass 08/50 completed | Min: 0.000019 | Max: 0.963652 | Mean: 0.295658
Pass 09/50 completed | Min: 0.000011 | Max: 0.969387 | Mean: 0.296028
Pass 10/50 completed | Min: 0.000002 | Max: 0.966035 | Mean: 0.295938
Pass 11/50 completed | Min: 0.000013 | Max: 0.987431 | Mean: 0.295665
Pass 12/50 completed | Min: 0.000011 | Max: 0.977493 | Mean: 0.296092
Pass 13/50 completed | Min: 0.000012 | Max: 0.967662 | Mean: 

In [70]:
# ============================================================
# MC DROPOUT UNCERTAINTY CALCULATION
# ============================================================

# mc_predictions shape:
# (50 stochastic passes, 38052 patients)

# ------------------------------------------------------------
# Mean prediction for every patient
# ------------------------------------------------------------

mc_mean = np.mean(
    mc_predictions,
    axis=0
)

# ------------------------------------------------------------
# Predictive variance for every patient
# ------------------------------------------------------------

mc_variance = np.var(
    mc_predictions,
    axis=0
)

# ------------------------------------------------------------
# Predictive standard deviation
# ------------------------------------------------------------

mc_std = np.sqrt(
    mc_variance
)

# ============================================================
# SANITY CHECK
# ============================================================

print("MC DROPOUT UNCERTAINTY CALCULATED")
print("=" * 60)

print("MC prediction shape:")
print(mc_predictions.shape)

print("\nMean prediction shape:")
print(mc_mean.shape)

print("\nVariance shape:")
print(mc_variance.shape)

print("\nStandard deviation shape:")
print(mc_std.shape)

print("\nExpected patient count:")
print(len(test_true_labels))

# ============================================================
# SUMMARY STATISTICS
# ============================================================

print("\nMEAN PREDICTION")
print("-" * 60)
print(f"Minimum : {mc_mean.min():.6f}")
print(f"Maximum : {mc_mean.max():.6f}")
print(f"Mean    : {mc_mean.mean():.6f}")
print(f"Median  : {np.median(mc_mean):.6f}")

print("\nPREDICTIVE VARIANCE")
print("-" * 60)
print(f"Minimum : {mc_variance.min():.10f}")
print(f"Maximum : {mc_variance.max():.10f}")
print(f"Mean    : {mc_variance.mean():.10f}")
print(f"Median  : {np.median(mc_variance):.10f}")

print("\nPREDICTIVE STANDARD DEVIATION")
print("-" * 60)
print(f"Minimum : {mc_std.min():.6f}")
print(f"Maximum : {mc_std.max():.6f}")
print(f"Mean    : {mc_std.mean():.6f}")
print(f"Median  : {np.median(mc_std):.6f}")

# ============================================================
# FIRST 10 PATIENTS
# ============================================================

print("\nFIRST 10 PATIENTS")
print("-" * 60)

for i in range(10):

    print(
        f"Patient {i+1:02d} | "
        f"Mean = {mc_mean[i]:.6f} | "
        f"Variance = {mc_variance[i]:.10f} | "
        f"Std = {mc_std[i]:.6f} | "
        f"True label = {int(test_true_labels[i])}"
    )

MC DROPOUT UNCERTAINTY CALCULATED
MC prediction shape:
(50, 38052)

Mean prediction shape:
(38052,)

Variance shape:
(38052,)

Standard deviation shape:
(38052,)

Expected patient count:
38052

MEAN PREDICTION
------------------------------------------------------------
Minimum : 0.000304
Maximum : 0.924065
Mean    : 0.295743
Median  : 0.239023

PREDICTIVE VARIANCE
------------------------------------------------------------
Minimum : 0.0000002760
Maximum : 0.0145257190
Mean    : 0.0015012044
Median  : 0.0014068999

PREDICTIVE STANDARD DEVIATION
------------------------------------------------------------
Minimum : 0.000525
Maximum : 0.120523
Mean    : 0.035931
Median  : 0.037509

FIRST 10 PATIENTS
------------------------------------------------------------
Patient 01 | Mean = 0.546721 | Variance = 0.0019651442 | Std = 0.044330 | True label = 1
Patient 02 | Mean = 0.113666 | Variance = 0.0007691144 | Std = 0.027733 | True label = 0
Patient 03 | Mean = 0.525235 | Variance = 0.003919560